<a href="https://colab.research.google.com/github/ADRIANVM117/data-science-portfolio/blob/main/scenario-portfolio-lab/notebooks/06_portfolio_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

El botón abre esta notebook en Google Colab. Allí ejecuta primero la siguiente celda de preparación para obtener el código y los reportes públicos; no descarga datos de mercado ni solicita credenciales.

# 07. Evaluación temporal de las reglas de cartera

## ¿Qué habría ocurrido al construir y mantener carteras con estas reglas, utilizando únicamente la información admisible en cada decisión?

En 06 construimos una cartera candidata de mínimo CVaR en un único corte histórico. Ese resultado describe su comportamiento bajo los escenarios utilizados para optimizarla, no su desempeño posterior.

En 07 reconstruiremos decisiones sucesivas y mediremos lo que ocurre después de cada una. Compararemos tres reglas: pesos iguales, mínimo CVaR con probabilidades exponenciales y mínimo CVaR condicionado por VIX.

Conservaremos los ocho ETFs y las restricciones de 06: cartera plenamente invertida, sin posiciones cortas, margen ni efectivo, y un tope de 15% por instrumento. Los escenarios de riesgo serán inicialmente de una sesión; el calendario de rebalanceo y la convención de ejecución se definirán por separado.

El ejercicio será retrospectivo. El universo se seleccionó después de observar la historia y la muestra ya fue explorada, por lo que no representa un *holdout* intacto ni resultados de operación real.

**Siguiente pregunta:** ¿Cuándo calculamos los pesos, cuándo podemos ejecutarlos y desde qué momento les atribuimos resultados?

## 2. ¿Cuándo decidimos y desde cuándo cuentan los resultados?

Todos los desplazamientos de esta convención corresponden a sesiones del calendario publicado, no a días naturales. Separamos la información que sirve para decidir de los retornos que atribuimos después a la decisión.

| Momento | Información o acción |
| --- | --- |
| Hasta el cierre de $t-1$ | Últimos retornos utilizados para ajustar el modelo. |
| Antes de ejecutar en $t$ | Calculamos los pesos con esa historia y el VIX de $t-1$. |
| Al cierre de $t$ | Ejecutamos hipotéticamente el rebalanceo. |
| Del cierre de $t$ al cierre de $t+1$ | Primer retorno atribuible a los pesos nuevos. |

Ejecutar al cierre es un supuesto del backtest, no una garantía de obtener ese precio en una operación real.

El retorno fechado $t$ pertenece a los pesos anteriores. En la primera inversión, la evaluación comienza después de ejecutar la cartera; todavía no calculamos resultados del periodo anterior a esa inversión.

No usamos el retorno ni el VIX de $t$ para decidir los pesos ejecutados al cierre de $t$.

Consideramos disponible el VIX de $t-1$ antes de ejecutar en $t$, siguiendo la evidencia aportada por el proveedor: normalmente estaría disponible hacia las 22:00 US/Eastern. Esta es una convención retrospectiva basada en esa comunicación, no una disponibilidad histórica verificada para cada fecha ni una garantía de publicación.

El VIX tiene dos usos distintos. En la muestra histórica de ajuste, cada retorno fechado $x$ conserva su asociación con el VIX de $x-2$, según la convención de 04/05. El VIX de $t-1$ identifica el estado conocido para la decisión actual. No cambiaremos la alineación histórica para hacer coincidir ambas fechas.

Las tres reglas —pesos iguales, mínimo CVaR exponencial y mínimo CVaR condicionado por VIX— compartirán la misma convención de decisión, ejecución y atribución de resultados.

Como ejemplo conceptual, considere tres sesiones consecutivas. En la sesión anterior termina la información utilizada; en la sesión de ejecución aplicamos los nuevos pesos al cierre; en la sesión siguiente observamos el primer retorno de los nuevos pesos. Este ejemplo no presupone precios, pesos ni rendimientos concretos.

**Siguiente pregunta:** ¿En qué sesiones rebalancearemos y cuánta historia exigiremos antes de invertir?

## 3. ¿Cuándo rebalanceamos y cuánta historia necesitamos?

### Frecuencia mensual

Para las tres reglas, el rebalanceo se ejecutará hipotéticamente al cierre de la última sesión publicada de cada mes calendario completo. La decisión utiliza información hasta la sesión anterior a la ejecución, según la convención temporal anterior. Identificaremos esas fechas con el calendario publicado, sin suponer que todos los meses terminan en un día hábil. El último día disponible del corte no se tratará automáticamente como cierre mensual: octubre de 2026 es parcial.

La frecuencia mensual es una decisión operativa inicial, no una frecuencia demostrada como óptima.

### Historia mínima

Exigiremos al menos 504 observaciones históricas alineadas y admisibles para el modelo condicionado por VIX. El mínimo CVaR exponencial utilizará exactamente esas mismas fechas de ajuste. Las 504 observaciones son filas conjuntas de retornos y estado histórico asociado, no días naturales.

La ventana será expansiva: comienza en la primera observación alineada disponible y crece en cada rebalanceo. La primera inversión ocurrirá en el primer cierre mensual que cumpla este requisito; todavía no calculamos cuál es esa fecha. Las 504 observaciones son un requisito de diseño, no una garantía de precisión.

### Parámetros fijos

Mantendremos una semivida exponencial de 1,102 sesiones, CVaR al 95% y un horizonte de riesgo de una sesión, junto con los ocho ETFs y las restricciones de 06. En cada rebalanceo condicionado reconstruiremos los cortes de VIX y los objetivos de *entropy pooling* con la muestra admisible de esa fecha. No reutilizaremos cortes ni probabilidades ajustadas con toda la historia de 04, ni buscaremos parámetros retrospectivamente para mejorar resultados.

Optimizar riesgo de una sesión y rebalancear mensualmente son decisiones distintas: no estamos minimizando directamente el CVaR mensual.

### Entre rebalanceos

No restauraremos diariamente los pesos objetivo. Los pesos evolucionarán con los retornos de los instrumentos. Pesos iguales significa volver a 12.5% por ETF en cada rebalanceo, no mantener 12.5% todos los días.

El máximo de 15% se exigirá sobre los pesos objetivo al rebalancear. La deriva posterior puede superar ese nivel sin activar operaciones adicionales. Tras el último rebalanceo, mantendremos las posiciones hasta el corte disponible, aunque el periodo final sea parcial.

**Siguiente pregunta:** ¿Podemos reconstruir estas fechas y sus muestras admisibles sin utilizar información futura?

## 4. ¿Qué fechas podemos usar para rebalancear?

Construimos los cierres mensuales con el calendario publicado y, para cada uno, identificamos la muestra expansiva disponible hasta la sesión anterior. Ambos modelos optimizados compartirán esas filas. El último mes del corte, octubre de 2026, es parcial y queda en la auditoría como no elegible. El horizonte de riesgo es una sesión y la semivida, 1,102 sesiones.

El raw VIX se verifica con el hash de su captura y la alineación con el hash de su manifiesto. Los hashes de los dos Parquet de matrices se comparan con los registrados posteriormente al crear la alineación: el manifiesto original del corte no ofrece hashes históricos de referencia para ellos. El catálogo tampoco tiene aquí un hash de referencia; verificamos sus identificadores y estados. La disponibilidad histórica del VIX sigue siendo una convención retrospectiva, no una comprobación point-in-time.

In [1]:
from pathlib import Path
from datetime import date
import hashlib, json, site
import numpy as np

def project_root():
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / 'scenario-portfolio-lab', parent / 'data-science-portfolio' / 'scenario-portfolio-lab'):
            if (candidate / 'config' / 'universe.csv').is_file() and (candidate / 'src' / 'scenario_portfolio_lab').is_dir():
                return candidate
    raise RuntimeError('Ejecute la notebook dentro del repositorio completo scenario-portfolio-lab.')

ROOT = project_root()
project_site = ROOT / '.venv' / 'Lib' / 'site-packages'
if project_site.is_dir():
    site.addsitedir(str(project_site))
import polars as pl

RUN = ROOT / 'data/published/cut_2026-10-09_run_2026-10-11_full_window_final'
ALIGNMENT = ROOT / 'data/published/vix_state_alignment_2026-10-31'
VIX_CAPTURE = ROOT / 'data/raw/vix_index_access_2026-10-29'
price_path = RUN / 'gold/adjusted_close_matrix.parquet'
return_path = RUN / 'gold/eligible_simple_total_return_matrix.parquet'
catalog_path = RUN / 'reports/eligible_universe.csv'
audit_path = ALIGNMENT / 'vix_return_alignment.parquet'
vix_path = VIX_CAPTURE / 'raw/VIX_INDEX_DATA_daily.json'
run_manifest = json.loads((RUN / 'run_manifest.json').read_text(encoding='utf-8'))
alignment_manifest = json.loads((ALIGNMENT / 'alignment_manifest.json').read_text(encoding='utf-8'))
vix_manifest = json.loads((VIX_CAPTURE / 'manifest.json').read_text(encoding='utf-8'))
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
integrity = {
    'VIX raw: hash de captura': sha256(vix_path) == vix_manifest['raw_sha256'] == alignment_manifest['inputs']['vix_raw']['sha256_capture_manifest_reference'],
    'Alineación: hash de su manifiesto': sha256(audit_path) == alignment_manifest['outputs']['alignment_table']['sha256'],
    'Precios: hash posterior de alineación': sha256(price_path) == alignment_manifest['inputs']['price_calendar']['sha256_computed_now'],
    'Retornos: hash posterior de alineación': sha256(return_path) == alignment_manifest['inputs']['simple_return_matrix']['sha256_computed_now'],
}
if not all(integrity.values()):
    raise AssertionError([name for name, ok in integrity.items() if not ok])

# Validamos los esquemas reales antes de seleccionar las columnas de trabajo.
price_schema = pl.read_parquet_schema(price_path); return_schema = pl.read_parquet_schema(return_path)
audit_schema = pl.read_parquet_schema(audit_path)
catalog = pl.read_csv(catalog_path)
required_audit = {'return_row_id', 'return_date', 'return_start_date', 'vix_source_date', 'vix_close', 'alignment_status'}
if list(price_schema)[0] != 'session_date' or list(return_schema)[0] != 'session_date' or not required_audit.issubset(audit_schema) or not {'symbol', 'asset_class', 'universe_status'}.issubset(catalog.columns):
    raise AssertionError('Esquema de entrada inesperado; revisar archivos de origen.')
eligible = catalog.filter(pl.col('universe_status') == 'ELIGIBLE')
eligible_symbols = eligible['symbol'].to_list()
SYMBOLS = ['SPY', 'VEA', 'EEM', 'TLT', 'BIL', 'GLD', 'USO', 'VNQ']
if len(eligible_symbols) != 125 or len(set(eligible_symbols)) != 125 or list(return_schema)[1:] != eligible_symbols:
    raise AssertionError('El catálogo elegible no coincide exactamente con las columnas de retornos.')
if any(eligible_symbols.count(symbol) != 1 or symbol not in price_schema for symbol in SYMBOLS):
    raise AssertionError('Falta o se repite un ETF en los insumos publicados.')
prices = pl.read_parquet(price_path, columns=['session_date', *SYMBOLS])
returns = pl.read_parquet(return_path, columns=['session_date', *SYMBOLS])
audit = pl.read_parquet(audit_path)
calendar = prices['session_date'].to_list()
cut = date.fromisoformat(run_manifest['approved_period']['end'])
start = date.fromisoformat(run_manifest['approved_period']['start'])
raw_vix = json.loads(vix_path.read_text(encoding='utf-8'))
if raw_vix.get('symbol') != 'VIX' or raw_vix.get('interval') != 'daily':
    raise AssertionError('El raw no identifica la serie VIX diaria.')
vix_records = [(date.fromisoformat(row['date']), float(row['close'])) for row in raw_vix['data']]
if len({d for d, _ in vix_records}) != len(vix_records):
    raise AssertionError('El raw VIX contiene fechas duplicadas.')
# No usamos los registros VIX posteriores al corte, aunque se conservan en el raw original.
vix_by_date = {d: value for d, value in vix_records if d <= cut}
if len(calendar) != 2205 or calendar[0] != start or calendar[-1] != cut or returns['session_date'].to_list() != calendar:
    raise AssertionError('El calendario de matrices no coincide con la ventana publicada.')
if calendar != sorted(calendar) or len(set(calendar)) != len(calendar):
    raise AssertionError('Las fechas del calendario no son únicas y crecientes.')
if returns.head(1).select(SYMBOLS).null_count().row(0) != (1,) * len(SYMBOLS):
    raise AssertionError('La primera fila de retornos debe ser nula por ausencia de precio anterior.')
return_values = returns.slice(1).select(SYMBOLS).to_numpy()
price_values = prices.select(SYMBOLS).to_numpy()
if not np.isfinite(return_values).all() or np.any(return_values <= -1) or not np.isfinite(price_values).all() or np.any(price_values <= 0):
    raise AssertionError('Hay retornos o precios inválidos en los ocho ETF.')

audit_rows = audit.to_dicts()
if len(audit_rows) != 2204 or [row['return_date'] for row in audit_rows] != calendar[1:]:
    raise AssertionError('La auditoría no conserva todas las fechas de retorno en orden.')
if audit_rows[0]['alignment_status'] != 'UNALIGNABLE_PRE_PUBLISHED_CALENDAR_START' or any(row['alignment_status'] != 'ALIGNED' for row in audit_rows[1:]):
    raise AssertionError('Estados de alineación inesperados; no se descartan filas silenciosamente.')
for i, row in enumerate(audit_rows):
    if row['return_row_id'] != i or row['return_start_date'] != calendar[i]:
        raise AssertionError(f'Identidad o inicio de retorno incorrecto: {row}')
    if i >= 1 and (row['vix_source_date'] != calendar[i-1] or row['vix_source_date'] not in vix_by_date or row['vix_close'] != vix_by_date[row['vix_source_date']] or not np.isfinite(row['vix_close']) or row['vix_close'] <= 0):
        raise AssertionError(f'Alineación histórica VIX x-2 inválida: {row}')
aligned_rows = audit_rows[1:]
aligned_dates = [row['return_date'] for row in aligned_rows]

# Un mes queda completo sólo si el calendario publicado alcanza el mes siguiente; octubre final es parcial.
month_ends = [i for i, day in enumerate(calendar) if i == len(calendar)-1 or (calendar[i+1].year, calendar[i+1].month) != (day.year, day.month)]
M, TAU, RISK_HORIZON = 504, 1102, 1
rebalance_rows = []
for i in month_ends:
    execution = calendar[i]; previous = calendar[i-1] if i > 0 else None
    fit = [d for d in aligned_dates if previous is not None and d <= previous]
    decision_vix = vix_by_date.get(previous)
    next_return = calendar[i+1] if i+1 < len(calendar) else None
    reasons = []
    if i == len(calendar)-1: reasons.append('MES_FINAL_PARCIAL')
    if len(fit) < M: reasons.append('HISTORIA_ALINEADA_INSUFICIENTE')
    if decision_vix is None or not np.isfinite(decision_vix) or decision_vix <= 0: reasons.append('VIX_DECISIÓN_NO_DISPONIBLE')
    if next_return is None: reasons.append('SIN_SESIÓN_POSTERIOR_EN_EL_CORTE')
    rebalance_rows.append({'fecha_ejecución': execution, 'fin_información': previous, 'inicio_ajuste': fit[0] if fit else None, 'fin_ajuste': fit[-1] if fit else None, 'observaciones': len(fit), 'fecha_vix_decisión': previous, 'vix_decisión': decision_vix, 'primer_retorno_nuevo': next_return, 'elegible': not reasons, 'motivo': ' | '.join(reasons) if reasons else 'ELEGIBLE'})
rebalance_audit = pl.DataFrame(rebalance_rows)
eligible_rebalances = rebalance_audit.filter(pl.col('elegible'))

# Comprobamos identidad, fechas y ventanas mediante posiciones, sin imputar ni ajustar filas.
positions = {day: i for i, day in enumerate(calendar)}
eligible_rows = eligible_rebalances.to_dicts()
reason_counts = {}
for row in rebalance_rows:
    if not row['elegible']:
        for reason in row['motivo'].split(' | '): reason_counts[reason] = reason_counts.get(reason, 0) + 1
controls = {**integrity,
    'Fechas únicas y crecientes': calendar == sorted(set(calendar)) and aligned_dates == sorted(set(aligned_dates)),
    'Ocho ETF en orden y elegibles': returns.columns == ['session_date', *SYMBOLS] and prices.columns == ['session_date', *SYMBOLS] and all(eligible_symbols.count(s) == 1 for s in SYMBOLS),
    'Cierres completos; octubre parcial excluido': all(positions[r['fecha_ejecución']] in month_ends[:-1] for r in eligible_rows) and rebalance_rows[-1]['motivo'].startswith('MES_FINAL_PARCIAL') and (cut.year, cut.month) == (2026, 10),
    'Ajuste anterior y conteo directo': all(r['fin_ajuste'] <= r['fin_información'] < r['fecha_ejecución'] and r['observaciones'] == sum(d <= r['fin_información'] for d in aligned_dates) for r in eligible_rows),
    'Mínimo y ventanas expansivas anidadas': all(r['observaciones'] >= M and r['inicio_ajuste'] == aligned_dates[0] for r in eligible_rows) and all(a['observaciones'] < b['observaciones'] for a, b in zip(eligible_rows, eligible_rows[1:])),
    'Alineación histórica x-2 y raw exacto': len(aligned_rows) == 2203 and all(r['vix_source_date'] == calendar[positions[r['return_date']]-2] and r['vix_close'] == vix_by_date[r['vix_source_date']] for r in aligned_rows),
    'VIX decisión en sesión anterior': all(r['fecha_vix_decisión'] == calendar[positions[r['fecha_ejecución']]-1] and r['vix_decisión'] == vix_by_date[r['fecha_vix_decisión']] for r in eligible_rows),
    'Primer retorno es sesión siguiente': all(r['primer_retorno_nuevo'] == calendar[positions[r['fecha_ejecución']]+1] and r['primer_retorno_nuevo'] > r['fecha_ejecución'] for r in eligible_rows),
    'Sin insumos posteriores al corte': all(d <= cut for d in calendar) and all(r['vix_source_date'] <= cut for r in aligned_rows) and all(r['fin_información'] <= cut for r in eligible_rows),
}
if not all(controls.values()):
    raise AssertionError([name for name, ok in controls.items() if not ok])
display(pl.DataFrame({'símbolo': SYMBOLS, 'clase': [eligible.filter(pl.col('symbol') == s)['asset_class'][0] for s in SYMBOLS]}))
print(f'Sesiones: {len(calendar)}; retornos: {len(calendar)-1}; filas alineadas: {len(aligned_rows)}; ETF: {len(SYMBOLS)}; M={M}; semivida={TAU}; horizonte={RISK_HORIZON}.')
print(f'Rebalanceos elegibles: {eligible_rebalances.height}; primer y último grupo de cinco:')
display(eligible_rebalances.head(5)); display(eligible_rebalances.tail(5))
display(pl.DataFrame({'motivo': list(reason_counts), 'conteo': list(reason_counts.values())}))
display(pl.DataFrame({'control': list(controls), 'resultado': ['PASS' if ok else 'FAIL' for ok in controls.values()]}))


símbolo,clase
str,str
"""SPY""","""etf"""
"""VEA""","""etf"""
"""EEM""","""etf"""
"""TLT""","""etf"""
"""BIL""","""etf"""
"""GLD""","""etf"""
"""USO""","""etf"""
"""VNQ""","""etf"""


Sesiones: 2205; retornos: 2204; filas alineadas: 2203; ETF: 8; M=504; semivida=1102; horizonte=1.
Rebalanceos elegibles: 81; primer y último grupo de cinco:


fecha_ejecución,fin_información,inicio_ajuste,fin_ajuste,observaciones,fecha_vix_decisión,vix_decisión,primer_retorno_nuevo,elegible,motivo
date,date,date,date,i64,date,f64,date,bool,str
2020-01-31,2020-01-30,2018-01-04,2020-01-30,521,2020-01-30,15.49,2020-02-03,true,"""ELEGIBLE"""
2020-02-28,2020-02-27,2018-01-04,2020-02-27,540,2020-02-27,39.16,2020-03-02,true,"""ELEGIBLE"""
2020-03-31,2020-03-30,2018-01-04,2020-03-30,562,2020-03-30,57.08,2020-04-01,true,"""ELEGIBLE"""
2020-04-30,2020-04-29,2018-01-04,2020-04-29,583,2020-04-29,31.23,2020-05-01,true,"""ELEGIBLE"""
2020-05-29,2020-05-28,2018-01-04,2020-05-28,603,2020-05-28,28.59,2020-06-01,true,"""ELEGIBLE"""


fecha_ejecución,fin_información,inicio_ajuste,fin_ajuste,observaciones,fecha_vix_decisión,vix_decisión,primer_retorno_nuevo,elegible,motivo
date,date,date,date,i64,date,f64,date,bool,str
2026-05-29,2026-05-28,2018-01-04,2026-05-28,2110,2026-05-28,15.74,2026-06-01,true,"""ELEGIBLE"""
2026-06-30,2026-06-29,2018-01-04,2026-06-29,2131,2026-06-29,17.65,2026-07-01,true,"""ELEGIBLE"""
2026-07-31,2026-07-30,2018-01-04,2026-07-30,2153,2026-07-30,17.09,2026-08-03,true,"""ELEGIBLE"""
2026-08-31,2026-08-28,2018-01-04,2026-08-28,2174,2026-08-28,14.43,2026-09-01,true,"""ELEGIBLE"""
2026-09-30,2026-09-29,2018-01-04,2026-09-29,2195,2026-09-29,16.04,2026-10-01,true,"""ELEGIBLE"""


motivo,conteo
str,i64
"""HISTORIA_ALINEADA_INSUFICIENTE""",24
"""MES_FINAL_PARCIAL""",1
"""SIN_SESIÓN_POSTERIOR_EN_EL_COR…",1


control,resultado
str,str
"""VIX raw: hash de captura""","""PASS"""
"""Alineación: hash de su manifie…","""PASS"""
"""Precios: hash posterior de ali…","""PASS"""
"""Retornos: hash posterior de al…","""PASS"""
"""Fechas únicas y crecientes""","""PASS"""
…,…
"""Mínimo y ventanas expansivas a…","""PASS"""
"""Alineación histórica x-2 y raw…","""PASS"""
"""VIX decisión en sesión anterio…","""PASS"""


## 5. ¿Qué carteras obtendríamos en el primer rebalanceo?

Con la primera fecha elegible reconstruimos la misma muestra admisible para las tres carteras: pesos iguales y mínimo CVaR al 95% bajo probabilidades exponenciales o condicionadas por el VIX previo. El ajuste condicionado estima sus cortes y momentos sólo con esa historia. Comparamos cada cartera bajo ambas distribuciones de ajuste; esta comparación describe los escenarios empleados, no resultados posteriores.

Las optimizaciones usan retornos simples conjuntos a una sesión, inversión completa y pesos entre 0% y 15%. La referencia exponencial tiene semivida de 1,102 sesiones. Para entropy pooling usamos residuos estandarizados: tolerancias de igualdad y factibilidad $10^{-9}$, KKT $10^{-8}$ y normalización $10^{-12}$. Para la programación lineal usamos $10^{-9}$ en factibilidad de pesos y $10^{-8}$ en conciliación del objetivo y comparación de CVaR; estos valores están en unidades decimales de retorno.

In [2]:
from scipy.optimize import linprog, minimize, root, least_squares
from scipy.special import logsumexp
from scipy.sparse import hstack, csr_matrix, eye
import plotly.graph_objects as go

first_rebalance = eligible_rebalances.row(0, named=True)
execution_date = first_rebalance['fecha_ejecución']
fit_end = first_rebalance['fin_información']
fit_audit = audit.filter((pl.col('alignment_status') == 'ALIGNED') & (pl.col('return_date') <= fit_end))
fit_dates = fit_audit['return_date'].to_list()
fit_returns_frame = returns.filter(pl.col('session_date').is_in(fit_dates))
fit_returns = fit_returns_frame.select(SYMBOLS).to_numpy().astype(float)
fit_vix = fit_audit['vix_close'].to_numpy().astype(float)
if not (execution_date == date(2020, 1, 31) and fit_end == date(2020, 1, 30) and len(fit_dates) == first_rebalance['observaciones'] == 521 and fit_returns_frame['session_date'].to_list() == fit_dates and fit_dates[0] == first_rebalance['inicio_ajuste'] and fit_dates[-1] == first_rebalance['fin_ajuste'] and fit_returns.shape == (521, 8)):
    raise AssertionError('La muestra del primer rebalanceo no coincide con el calendario.')

# Las edades se miden en posiciones del calendario de sesiones, respecto de la última fila de ajuste.
fit_positions = np.array([positions[d] for d in fit_dates])
fit_ages = fit_positions[-1] - fit_positions
p_exp = np.exp(-np.log(2.0) * fit_ages / TAU); p_exp /= p_exp.sum()
sorted_vix = np.sort(fit_vix)
vix_p25 = float(sorted_vix[int(np.ceil(len(fit_vix) * 0.25)) - 1])
vix_p75 = float(sorted_vix[int(np.ceil(len(fit_vix) * 0.75)) - 1])
if not vix_p25 < vix_p75: raise AssertionError('Los cortes VIX no son distintos.')
fit_groups = np.where(fit_vix <= vix_p25, 1, np.where(fit_vix <= vix_p75, 2, 3))
decision_vix = float(first_rebalance['vix_decisión'])
selected_group = 1 if decision_vix <= vix_p25 else (2 if decision_vix <= vix_p75 else 3)
group_values = fit_vix[fit_groups == selected_group]
if len(group_values) == 0: raise AssertionError('El grupo VIX seleccionado está vacío.')
target_mean = float(group_values.mean()); target_variance = float(np.mean((group_values - target_mean)**2))
if target_variance <= 0: raise AssertionError('La dispersión VIX crisp debe ser positiva.')
target_std = np.sqrt(target_variance); target_second = target_mean**2 + target_variance
a = (fit_vix - target_mean) / target_std; b = a**2; log_p = np.log(p_exp)

# Dual convexo: primero probamos la solución con eta=0; si no es factible, resolvemos la restricción activa.
def entropy_dual(theta):
    log_unscaled = log_p - theta[0] * a - theta[1] * b
    log_q = log_unscaled - logsumexp(log_unscaled); q = np.exp(log_q)
    return float(logsumexp(log_unscaled) + theta[1]), np.array([-q @ a, 1.0 - q @ b]), q, log_q

def entropy_hessian(theta):
    q = entropy_dual(theta)[2]; features = np.column_stack((a, b)); centered = features - q @ features
    return (centered * q[:, None]).T @ centered

def solve_entropy_dual(dual, hessian):
    boundary = root(lambda x: [dual(np.array([x[0], 0.0]))[1][0]], [0.0], method='hybr', options={'xtol': 1e-12})
    if not boundary.success: raise RuntimeError(f'Entropy pooling, frontera: {boundary.message}')
    boundary_theta = np.array([boundary.x[0], 0.0])
    if dual(boundary_theta)[1][1] >= -1e-9:
        return boundary_theta, 'INACTIVA'
    result = minimize(lambda x: dual(x)[0], [boundary_theta[0], 0.1], jac=lambda x: dual(x)[1], method='SLSQP', bounds=[(None, None), (0.0, None)], options={'ftol': 1e-12, 'maxiter': 10000})
    if not result.success: raise RuntimeError(f'Entropy pooling, activo: {result.message}')
    polished = root(lambda x: dual(x)[1], result.x, jac=hessian, method='hybr', options={'xtol': 1e-12})
    candidate = polished.x
    if not polished.success or candidate[1] <= 0 or np.max(np.abs(dual(candidate)[1])) > 1e-8:
        refined = least_squares(lambda x: dual(x)[1], result.x, bounds=([-np.inf, 0.0], [np.inf, np.inf]), xtol=1e-14, ftol=1e-14, gtol=1e-14, max_nfev=10000)
        if not refined.success: raise RuntimeError(f'Entropy pooling, pulido: {refined.message}')
        candidate = refined.x
    if candidate[1] <= 0: raise AssertionError('El multiplicador activo debe ser positivo.')
    return candidate, 'ACTIVA'

theta, ep_case = solve_entropy_dual(entropy_dual, entropy_hessian)
_, gradient, q_vix, log_q_vix = entropy_dual(theta)
eta = float(theta[1]); slack = float(gradient[1]); kl = float(q_vix @ (log_q_vix - log_p))
fd_point = np.array([0.17, 0.23]); fd_step = 1e-6
fd_gradient = np.array([(entropy_dual(fd_point + np.eye(2)[i] * fd_step)[0] - entropy_dual(fd_point - np.eye(2)[i] * fd_step)[0]) / (2 * fd_step) for i in range(2)])
gradient_error = float(np.max(np.abs(fd_gradient - entropy_dual(fd_point)[1])))

# CVaR: min z + sum(p*u)/0.05; u >= -(R@w)-z, u >= 0 y z libre.
scenario_count, asset_count = fit_returns.shape; tail = 0.05; cap = 0.15
constraint_matrix = hstack([csr_matrix(-fit_returns), csr_matrix(-np.ones((scenario_count, 1))), -eye(scenario_count, format='csr')], format='csr')
investment_matrix = csr_matrix(np.r_[np.ones(asset_count), np.zeros(1 + scenario_count)].reshape(1, -1))
bounds = [(0.0, cap)] * asset_count + [(None, None)] + [(0.0, None)] * scenario_count
def solve_cvar(probabilities, matrix=constraint_matrix, investment=investment_matrix, variable_bounds=bounds):
    objective = np.r_[np.zeros(asset_count), 1.0, probabilities / tail]
    result = linprog(objective, A_ub=matrix, b_ub=np.zeros(len(probabilities)), A_eq=investment, b_eq=[1.0], bounds=variable_bounds, method='highs')
    if not result.success: raise RuntimeError(f'CVaR HiGHS: {result.message}')
    return result

exp_solution = solve_cvar(p_exp); vix_solution = solve_cvar(q_vix)
portfolios = {'Pesos iguales': np.full(asset_count, 1.0 / asset_count), 'Mínimo CVaR exponencial': exp_solution.x[:asset_count], 'Mínimo CVaR VIX': vix_solution.x[:asset_count]}

# Ordenamos pérdidas y tomamos exactamente el 5% de masa; conciliamos con la identidad basada en VaR.
def tail_metrics(losses, probabilities):
    order = np.argsort(losses, kind='stable'); cumulative = np.cumsum(probabilities[order])
    var = float(losses[order[np.searchsorted(cumulative, 1.0 - tail, side='left')]])
    reverse = np.argsort(-losses, kind='stable'); p_sorted = probabilities[reverse]
    before = np.r_[0.0, np.cumsum(p_sorted[:-1])]
    included = np.minimum(np.maximum(tail - before, 0.0), p_sorted)
    cvar = float(included @ losses[reverse] / tail)
    independent = float(var + probabilities @ np.maximum(losses - var, 0.0) / tail)
    return cvar, independent, float(included.sum())

distributions = {'Exponencial': p_exp, 'Condicionada VIX': q_vix}
risk = {(name, distribution): tail_metrics(-(fit_returns @ weights), probabilities) for name, weights in portfolios.items() for distribution, probabilities in distributions.items()}
weight_table = pl.DataFrame({'ETF': SYMBOLS, **{name: 100 * weights for name, weights in portfolios.items()}})
risk_table = pl.DataFrame({'Cartera': list(portfolios), 'CVaR exponencial (%)': [100 * risk[(name, 'Exponencial')][0] for name in portfolios], 'CVaR condicionado VIX (%)': [100 * risk[(name, 'Condicionada VIX')][0] for name in portfolios]})

TOL = 1e-12; WEIGHT_TOL = 1e-9; CVAR_TOL = 1e-8
block5_controls = {
    'Muestra y alineación comunes': fit_returns_frame['session_date'].to_list() == fit_dates and all(r['vix_source_date'] == calendar[positions[r['return_date']]-2] and r['return_date'] <= fit_end for r in fit_audit.to_dicts()) and first_rebalance['fecha_vix_decisión'] == fit_end and decision_vix == vix_by_date[fit_end],
    'Cortes y grupos crisp': len(group_values) > 0 and np.all(np.isin(fit_groups, [1, 2, 3])) and fit_ages[-1] == 0 and np.all(np.diff(fit_ages) > 0) == False and np.all(np.diff(fit_ages) < 0),
    'Probabilidades normalizadas': all(np.isfinite(p).all() and np.all(p >= 0) and np.isclose(p.sum(), 1.0, atol=TOL, rtol=TOL) for p in distributions.values()),
    'Entropy pooling factible': abs(float(q_vix @ a)) <= 1e-9 and float(q_vix @ b) <= 1 + 1e-9 and np.isclose(q_vix @ fit_vix, target_mean, atol=1e-9 * target_std, rtol=0) and q_vix @ (fit_vix**2) <= target_second + 1e-9 * target_variance,
    'Entropy pooling KKT y gradiente': abs(gradient[0]) <= 1e-8 and eta >= 0 and slack >= -1e-8 and abs(eta * slack) <= 1e-8 and gradient_error <= 1e-7 and np.isfinite(kl) and kl >= -1e-12,
    'Pesos factibles sin corrección': all(np.isclose(w.sum(), 1, atol=WEIGHT_TOL, rtol=0) and np.all(w >= -WEIGHT_TOL) and np.all(w <= cap + WEIGHT_TOL) for w in portfolios.values()),
    'Cola exacta y CVaR independiente': all(np.isclose(mass, tail, atol=TOL, rtol=TOL) and np.isclose(cvar, independent, atol=TOL, rtol=TOL) for cvar, independent, mass in risk.values()),
    'Objetivos HiGHS concilian': abs(exp_solution.fun - risk[('Mínimo CVaR exponencial', 'Exponencial')][0]) <= CVAR_TOL and abs(vix_solution.fun - risk[('Mínimo CVaR VIX', 'Condicionada VIX')][0]) <= CVAR_TOL,
    'Óptimos no peores en su distribución': all(risk[('Mínimo CVaR exponencial', 'Exponencial')][0] <= risk[(name, 'Exponencial')][0] + CVAR_TOL and risk[('Mínimo CVaR VIX', 'Condicionada VIX')][0] <= risk[(name, 'Condicionada VIX')][0] + CVAR_TOL for name in portfolios),
    'Restricciones auxiliares de ambos LP': all(np.all(sol.x[asset_count+1:] >= -(fit_returns @ sol.x[:asset_count]) - sol.x[asset_count] - WEIGHT_TOL) and np.all(sol.x[asset_count+1:] >= -WEIGHT_TOL) for sol in (exp_solution, vix_solution)),
}
if not all(block5_controls.values()): raise AssertionError([name for name, ok in block5_controls.items() if not ok])

print(f'Rebalanceo: {execution_date}; ajuste: {fit_dates[0]} a {fit_end}; observaciones: {len(fit_dates)}.')
display(weight_table)
display(pl.DataFrame({'p25': [vix_p25], 'p75': [vix_p75], 'fecha VIX': [fit_end], 'VIX decisión': [decision_vix], 'grupo': [selected_group], 'n crisp': [len(group_values)], 'media objetivo': [target_mean], 'media obtenida': [float(q_vix @ fit_vix)], 'varianza límite': [target_variance], 'varianza obtenida': [float(q_vix @ ((fit_vix-target_mean)**2))], 'caso': [ep_case], 'eta': [eta], 'holgura': [slack], 'N_eff exp': [float(1 / (p_exp @ p_exp))], 'N_eff VIX': [float(1 / (q_vix @ q_vix))], 'KL': [kl], 'underflow': [int(np.count_nonzero(q_vix == 0))]}))
display(risk_table)
figure = go.Figure()
for name, weights in portfolios.items():
    figure.add_bar(name=name, y=SYMBOLS, x=100 * weights, orientation='h', hovertemplate='%{y}: %{x:.5f}%<extra>%{fullData.name}</extra>')
figure.add_vline(x=15, line_dash='dash', line_color='#444444')
figure.update_layout(title='Pesos objetivo del primer rebalanceo', barmode='group', paper_bgcolor='#FFF8E7', plot_bgcolor='#FFF8E7', font_color='#2D2A26', legend=dict(x=1.02, y=1), margin=dict(l=70, r=220, t=60, b=55))
figure.update_xaxes(title='Peso objetivo (%)', range=[0, 17], gridcolor='#E7DDCC'); figure.update_yaxes(categoryorder='array', categoryarray=list(reversed(SYMBOLS)))
figure.show()
display(pl.DataFrame({'Control': list(block5_controls), 'Resultado': ['PASS' if ok else 'FAIL' for ok in block5_controls.values()]}))
print(f'Residuos KKT: igualdad={gradient[0]:.3e}, holgura={slack:.3e}, complementariedad={eta*slack:.3e}; error gradiente={gradient_error:.3e}.')


Rebalanceo: 2020-01-31; ajuste: 2018-01-04 a 2020-01-30; observaciones: 521.


ETF,Pesos iguales,Mínimo CVaR exponencial,Mínimo CVaR VIX
str,f64,f64,f64
"""SPY""",12.5,15.0,15.0
"""VEA""",12.5,15.0,15.0
"""EEM""",12.5,4.450925,0.666044
"""TLT""",12.5,15.0,15.0
"""BIL""",12.5,15.0,15.0
"""GLD""",12.5,15.0,15.0
"""USO""",12.5,5.549075,9.333956
"""VNQ""",12.5,15.0,15.0


p25,p75,fecha VIX,VIX decisión,grupo,n crisp,media objetivo,media obtenida,varianza límite,varianza obtenida,caso,eta,holgura,N_eff exp,N_eff VIX,KL,underflow
f64,f64,date,f64,i64,i64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64
12.88,17.97,2020-01-30,15.49,2,260,15.031346,15.031346,2.045285,2.045285,"""ACTIVA""",0.457661,-2.2204e-16,516.387041,272.274201,0.480215,0


Cartera,CVaR exponencial (%),CVaR condicionado VIX (%)
str,f64,f64
"""Pesos iguales""",1.194557,1.088444
"""Mínimo CVaR exponencial""",0.948994,0.855785
"""Mínimo CVaR VIX""",0.958341,0.85095


Control,Resultado
str,str
"""Muestra y alineación comunes""","""PASS"""
"""Cortes y grupos crisp""","""PASS"""
"""Probabilidades normalizadas""","""PASS"""
"""Entropy pooling factible""","""PASS"""
"""Entropy pooling KKT y gradient…","""PASS"""
"""Pesos factibles sin corrección""","""PASS"""
"""Cola exacta y CVaR independien…","""PASS"""
"""Objetivos HiGHS concilian""","""PASS"""
"""Óptimos no peores en su distri…","""PASS"""


Residuos KKT: igualdad=1.561e-17, holgura=-2.220e-16, complementariedad=-1.016e-16; error gradiente=3.218e-10.


## 6. ¿Qué ocurre después de ejecutar las primeras carteras?

Seguimos cada cartera desde su ejecución inicial hasta el cierre del siguiente rebalanceo elegible, incluido. Durante ese intervalo mantenemos las posiciones: los pesos cambian con los retornos, sin restaurarse diariamente. Cada retorno observado se utiliza una vez en su fecha; las probabilidades históricas no ponderan estos resultados.

Los retornos simples de rendimiento total incorporan la convención de reinversión implícita del proveedor. Esta es una contabilidad de rentabilidad, no una reconstrucción de títulos o flujos reales de dividendos; tampoco usamos precios ajustados como precios ejecutables. No incluimos costos.

Para cada instrumento $j$ y sesión $d$, iniciamos con $V_{j,0}=V_0w_{j,0}$ y aplicamos:

$$V_{j,d}=V_{j,d-1}(1+r_{j,d}),\qquad V_d=\sum_j V_{j,d},\qquad w_{j,d}=V_{j,d}/V_d,\qquad r_{p,d}=\sum_j w_{j,d-1}r_{j,d}.$$

El capital inicial ilustrativo es USD 1,000,000 por regla. El drawdown se mide frente al máximo acumulado incluyendo ese capital inicial.

In [3]:
# Copias previas para comprobar que la contabilidad no altera los objetivos ni la distribución.
saved_weights = {name: weights.copy() for name, weights in portfolios.items()}
saved_p_exp = p_exp.copy(); saved_q_vix = q_vix.copy()
saved_calendar = tuple(calendar); saved_rebalances = eligible_rebalances.clone()
initial_capital = 1_000_000.0
initial_execution = eligible_rebalances['fecha_ejecución'][0]
next_execution = eligible_rebalances['fecha_ejecución'][1]
first_return_date = eligible_rebalances['primer_retorno_nuevo'][0]
if initial_execution != date(2020, 1, 31) or first_return_date != date(2020, 2, 3):
    raise AssertionError('La primera ejecución o sesión atribuible no coincide con el calendario validado.')
if not (execution_date == initial_execution < first_return_date <= next_execution):
    raise AssertionError('Las fechas de ejecución y seguimiento no son coherentes.')

period = returns.filter((pl.col('session_date') > initial_execution) & (pl.col('session_date') <= next_execution))
period_dates = period['session_date'].to_list()
period_returns = period.select(SYMBOLS).to_numpy().astype(float)
expected_dates = calendar[positions[initial_execution] + 1:positions[next_execution] + 1]
if period_dates != expected_dates or not period_dates or period_dates[0] != first_return_date or period_dates[-1] != next_execution:
    raise AssertionError('Las fechas del periodo no coinciden con las sesiones publicadas.')
if period.columns != ['session_date', *SYMBOLS] or not np.isfinite(period_returns).all() or np.any(period_returns <= -1):
    raise AssertionError('Retornos o símbolos del periodo inválidos.')

# Cada fila de valores es un cierre; la fila cero es el instante inicial de la inversión.
accounting = {}
for name, target_weights in portfolios.items():
    instrument_values = np.empty((len(period_dates) + 1, len(SYMBOLS)))
    instrument_values[0] = initial_capital * target_weights
    daily_portfolio_returns = np.empty(len(period_dates))
    for d, asset_returns in enumerate(period_returns, start=1):
        previous_weights = instrument_values[d-1] / instrument_values[d-1].sum()
        daily_portfolio_returns[d-1] = previous_weights @ asset_returns
        instrument_values[d] = instrument_values[d-1] * (1.0 + asset_returns)
    capital = instrument_values.sum(axis=1)
    closing_weights = instrument_values / capital[:, None]
    running_peak = np.maximum.accumulate(capital)
    drawdown = capital / running_peak - 1.0
    accounting[name] = {'values': instrument_values, 'capital': capital, 'weights': closing_weights, 'daily_returns': daily_portfolio_returns, 'drawdown': drawdown}

summary = pl.DataFrame([{'regla': name, 'capital_final_usd': item['capital'][-1], 'retorno_acumulado_%': 100 * (item['capital'][-1] / initial_capital - 1), 'máximo_drawdown_%': 100 * item['drawdown'].min(), 'superó_15%_por_deriva': bool(np.any(item['weights'][1:] > 0.15 + 1e-12))} for name, item in accounting.items()])
weight_rows = []
for name, item in accounting.items():
    for j, symbol in enumerate(SYMBOLS):
        weight_rows.append({'regla': name, 'ETF': symbol, 'peso_inicial_%': 100 * saved_weights[name][j], 'peso_al_cierre_%': 100 * item['weights'][-1, j], 'superó_15%_durante_periodo': bool(np.any(item['weights'][1:, j] > 0.15 + 1e-12))})
period_weight_table = pl.DataFrame(weight_rows)

# Las identidades se contrastan sin redondear. En dólares usamos tolerancia absoluta 1e-7 y relativa 1e-12.
amount_atol, numeric_rtol, numeric_atol = 1e-7, 1e-12, 1e-12
period_controls = {
    'Mismas sesiones y retornos para las tres reglas': period_dates == expected_dates and period_returns.shape == (len(expected_dates), len(SYMBOLS)) and len(accounting) == len(portfolios) == 3,
    'Primer retorno posterior a ejecución': period_dates[0] == first_return_date > initial_execution,
    'Último retorno incluido antes de rebalancear': period_dates[-1] == next_execution == eligible_rebalances['fecha_ejecución'][1],
    'Pesos iniciales preservados': all(np.array_equal(item['values'][0], initial_capital * saved_weights[name]) and np.allclose(item['weights'][0], saved_weights[name], atol=numeric_atol, rtol=numeric_rtol) for name, item in accounting.items()),
    'Valores finitos y positivos; pesos suman uno': all(np.isfinite(item['values']).all() and np.all(item['values'] >= 0) and np.all(item['capital'] > 0) and np.isfinite(item['weights']).all() and np.allclose(item['weights'].sum(axis=1), 1, atol=numeric_atol, rtol=numeric_rtol) for item in accounting.values()),
    'Retorno diario con pesos anteriores': all(np.allclose(item['daily_returns'], np.sum(item['weights'][:-1] * period_returns, axis=1), atol=numeric_atol, rtol=numeric_rtol) and np.allclose(item['capital'][1:] / item['capital'][:-1] - 1, item['daily_returns'], atol=numeric_atol, rtol=numeric_rtol) for item in accounting.values()),
    'Valores por ETF concilian con producto de factores': all(np.allclose(item['values'][1:], initial_capital * saved_weights[name] * np.cumprod(1 + period_returns, axis=0), atol=amount_atol, rtol=numeric_rtol) for name, item in accounting.items()),
    'Capital concilia con composición diaria': all(np.allclose(item['capital'][1:], initial_capital * np.cumprod(1 + item['daily_returns']), atol=amount_atol, rtol=numeric_rtol) for item in accounting.values()),
    'Drawdown incluye capital inicial': all(item['capital'][0] == initial_capital and item['drawdown'][0] == 0 and np.allclose(item['drawdown'], item['capital'] / np.maximum.accumulate(item['capital']) - 1, atol=numeric_atol, rtol=numeric_rtol) for item in accounting.values()),
    'Objetivos, probabilidades y calendario preservados': all(np.array_equal(portfolios[name], weights) for name, weights in saved_weights.items()) and np.array_equal(p_exp, saved_p_exp) and np.array_equal(q_vix, saved_q_vix) and tuple(calendar) == saved_calendar and eligible_rebalances.equals(saved_rebalances),
}
if not all(period_controls.values()):
    raise AssertionError([name for name, ok in period_controls.items() if not ok])

print(f'Ejecución inicial: {initial_execution}; primera sesión: {period_dates[0]}; cierre siguiente rebalanceo incluido: {next_execution}; sesiones: {len(period_dates)}.')
display(summary)
with pl.Config(tbl_rows=30):
    display(period_weight_table)
capital_figure = go.Figure()
for name, item in accounting.items():
    capital_figure.add_scatter(x=[initial_execution, *period_dates], y=item['capital'], mode='lines', name=name, hovertemplate='%{x|%Y-%m-%d}<br>Capital: USD %{y:,.2f}<extra>%{fullData.name}</extra>')
capital_figure.update_layout(title='Capital bruto entre los dos primeros rebalanceos', xaxis_title='Fecha de cierre', yaxis_title='Capital (USD)', paper_bgcolor='#FFF8E7', plot_bgcolor='#FFF8E7', font_color='#2D2A26', legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0), margin=dict(l=80, r=40, t=110, b=60))
capital_figure.update_xaxes(gridcolor='#E7DDCC'); capital_figure.update_yaxes(gridcolor='#E7DDCC')
capital_figure.show()
display(pl.DataFrame({'Control': list(period_controls), 'Resultado': ['PASS' if ok else 'FAIL' for ok in period_controls.values()]}))


Ejecución inicial: 2020-01-31; primera sesión: 2020-02-03; cierre siguiente rebalanceo incluido: 2020-02-28; sesiones: 19.


regla,capital_final_usd,retorno_acumulado_%,máximo_drawdown_%,superó_15%_por_deriva
str,f64,f64,f64,bool
"""Pesos iguales""",958645.234045,-4.135477,-6.759617,false
"""Mínimo CVaR exponencial""",966476.233418,-3.352377,-5.926497,true
"""Mínimo CVaR VIX""",963052.031754,-3.694797,-6.264971,true


regla,ETF,peso_inicial_%,peso_al_cierre_%,superó_15%_durante_periodo
str,str,f64,f64,bool
"""Pesos iguales""","""SPY""",12.5,12.006974,false
"""Pesos iguales""","""VEA""",12.5,12.041614,false
"""Pesos iguales""","""EEM""",12.5,12.546896,false
"""Pesos iguales""","""TLT""",12.5,13.903292,false
"""Pesos iguales""","""BIL""",12.5,13.054681,false
"""Pesos iguales""","""GLD""",12.5,12.956282,false
"""Pesos iguales""","""USO""",12.5,11.367229,false
"""Pesos iguales""","""VNQ""",12.5,12.123031,false
"""Mínimo CVaR exponencial""","""SPY""",15.0,14.291623,true


Control,Resultado
str,str
"""Mismas sesiones y retornos par…","""PASS"""
"""Primer retorno posterior a eje…","""PASS"""
"""Último retorno incluido antes …","""PASS"""
"""Pesos iniciales preservados""","""PASS"""
"""Valores finitos y positivos; p…","""PASS"""
"""Retorno diario con pesos anter…","""PASS"""
"""Valores por ETF concilian con …","""PASS"""
"""Capital concilia con composici…","""PASS"""
"""Drawdown incluye capital inici…","""PASS"""


Este periodo ilustra cómo se materializa una decisión y cómo derivan los pesos entre rebalanceos. Por sí solo no permite elegir un modelo ni demostrar superioridad. Los resultados son brutos y no incorporan costos.

## 7. ¿Qué necesitamos operar para rebalancear?

Para la segunda decisión reconstruimos las distribuciones y los pesos usando sólo información hasta la sesión anterior a la ejecución. Los valores que llegan al cierre incluyen el retorno de la sesión de ejecución y sirven para dimensionar operaciones hipotéticas, no para ajustar los objetivos. Ejecutar exactamente esos montos al precio de cierre sigue siendo una idealización del backtest.

Para cada ETF, el valor objetivo es el capital previo multiplicado por su peso objetivo; la operación es valor objetivo menos valor previo: positivo indica compra y negativo, venta. Sin aportaciones, retiros ni costos, las compras deben igualar las ventas.

La **fracción negociada** es el nocional total comprado más vendido dividido entre el capital. La **rotación unilateral** es la mitad de esa fracción, equivalente a $\frac12\sum_j|w_j^{objetivo}-w_j^{previo}|$. Son convenciones distintas; los costos futuros por lado se aplicarían a los importes efectivamente comprados y vendidos. Los importes de esta sección no son órdenes.

Para los controles usamos tolerancia absoluta de $10^{-9}$ para pesos y USD $10^{-6}$ para montos; para CVaR usamos $10^{-8}$ en retornos decimales.

In [4]:
# Conservamos la primera decisión y el periodo antes de preparar la segunda.
first_weight_snapshot = {name: w.copy() for name, w in portfolios.items()}
first_value_snapshot = {name: item['values'].copy() for name, item in accounting.items()}
first_probability_snapshot = (p_exp.copy(), q_vix.copy())
first_calendar_snapshot = eligible_rebalances.clone()
second_rebalance = eligible_rebalances.row(1, named=True)
second_execution = second_rebalance['fecha_ejecución']
second_fit_end = second_rebalance['fin_información']
second_audit = audit.filter((pl.col('alignment_status') == 'ALIGNED') & (pl.col('return_date') <= second_fit_end))
second_dates = second_audit['return_date'].to_list()
second_frame = returns.filter(pl.col('session_date').is_in(second_dates))
second_returns = second_frame.select(SYMBOLS).to_numpy().astype(float)
second_vix = second_audit['vix_close'].to_numpy().astype(float)
if second_execution != date(2020, 2, 28) or second_execution != next_execution or second_frame['session_date'].to_list() != second_dates or len(second_dates) != second_rebalance['observaciones']:
    raise AssertionError('El calendario y la segunda muestra no coinciden.')
if second_dates[:len(fit_dates)] != fit_dates or second_dates[-1] != second_fit_end or second_fit_end != calendar[positions[second_execution]-1]:
    raise AssertionError('La segunda ventana debe extender la primera hasta t-1.')

second_ages = positions[second_dates[-1]] - np.array([positions[d] for d in second_dates])
second_exp = np.exp(-np.log(2.0) * second_ages / TAU); second_exp /= second_exp.sum()
second_sorted_vix = np.sort(second_vix)
second_p25 = float(second_sorted_vix[int(np.ceil(len(second_dates) * 0.25)) - 1])
second_p75 = float(second_sorted_vix[int(np.ceil(len(second_dates) * 0.75)) - 1])
if not second_p25 < second_p75: raise AssertionError('Cortes VIX no distintos.')
second_groups = np.where(second_vix <= second_p25, 1, np.where(second_vix <= second_p75, 2, 3))
second_decision_vix = float(second_rebalance['vix_decisión'])
second_group = 1 if second_decision_vix <= second_p25 else (2 if second_decision_vix <= second_p75 else 3)
second_group_values = second_vix[second_groups == second_group]
if len(second_group_values) == 0: raise AssertionError('Grupo VIX vacío.')
second_mean = float(second_group_values.mean()); second_variance = float(np.mean((second_group_values - second_mean)**2))
if second_variance <= 0: raise AssertionError('Varianza crisp VIX no positiva.')
second_std = np.sqrt(second_variance); second_a = (second_vix - second_mean) / second_std; second_b = second_a**2
second_log_p = np.log(second_exp)
def second_dual(theta):
    log_unscaled = second_log_p - theta[0] * second_a - theta[1] * second_b
    log_q = log_unscaled - logsumexp(log_unscaled); q = np.exp(log_q)
    return float(logsumexp(log_unscaled) + theta[1]), np.array([-q @ second_a, 1.0 - q @ second_b]), q, log_q
def second_hessian(theta):
    q = second_dual(theta)[2]; features = np.column_stack((second_a, second_b)); centered = features - q @ features
    return (centered * q[:, None]).T @ centered
second_theta, second_case = solve_entropy_dual(second_dual, second_hessian)
_, second_gradient, second_q, second_log_q = second_dual(second_theta)
second_kl = float(second_q @ (second_log_q - second_log_p))

# La función extraída reproduce el primer ajuste y sus pesos, comparados también con las salidas guardadas antes de extraerla.
first_theta_again, first_case_again = solve_entropy_dual(entropy_dual, entropy_hessian)
first_q_again = entropy_dual(first_theta_again)[2]
first_exp_again = solve_cvar(p_exp).x[:asset_count]; first_vix_again = solve_cvar(first_q_again).x[:asset_count]
previous_exp_display = np.array([15, 15, 4.450925, 15, 15, 15, 5.549075, 15]) / 100
previous_vix_display = np.array([15, 15, 0.666044, 15, 15, 15, 9.333956, 15]) / 100
first_reproduced = first_case_again == ep_case and np.allclose(first_q_again, q_vix, atol=1e-12, rtol=1e-12) and np.allclose(first_exp_again, portfolios['Mínimo CVaR exponencial'], atol=1e-9, rtol=0) and np.allclose(first_vix_again, portfolios['Mínimo CVaR VIX'], atol=1e-9, rtol=0) and np.allclose(first_exp_again, previous_exp_display, atol=6e-9, rtol=0) and np.allclose(first_vix_again, previous_vix_display, atol=6e-9, rtol=0)

# Los dos LP comparten exactamente la misma matriz de 540 filas y el orden de los ocho ETFs.
second_n = len(second_dates)
second_constraint = hstack([csr_matrix(-second_returns), csr_matrix(-np.ones((second_n, 1))), -eye(second_n, format='csr')], format='csr')
second_investment = csr_matrix(np.r_[np.ones(asset_count), np.zeros(1 + second_n)].reshape(1, -1))
second_bounds = [(0.0, cap)] * asset_count + [(None, None)] + [(0.0, None)] * second_n
second_exp_solution = solve_cvar(second_exp, second_constraint, second_investment, second_bounds)
second_vix_solution = solve_cvar(second_q, second_constraint, second_investment, second_bounds)
second_targets = {'Pesos iguales': np.full(asset_count, 1.0 / asset_count), 'Mínimo CVaR exponencial': second_exp_solution.x[:asset_count], 'Mínimo CVaR VIX': second_vix_solution.x[:asset_count]}

# Los valores de cierre dimensionan las operaciones: no intervienen en el ajuste anterior.
money_tol = 1e-6; weight_tol = 1e-9
operations = {}; operation_rows = []; turnover_rows = []
for name, target in second_targets.items():
    before_values = accounting[name]['values'][-1].copy()
    capital_before = float(before_values.sum()); previous_weights = before_values / capital_before
    target_values = capital_before * target; delta = target_values - before_values
    buys = float(np.maximum(delta, 0).sum()); sells = float(np.maximum(-delta, 0).sum())
    gross = float(np.abs(delta).sum()); traded_fraction = gross / capital_before
    unilateral_turnover = float(0.5 * np.abs(target - previous_weights).sum())
    operations[name] = {'before': before_values, 'capital': capital_before, 'previous_weights': previous_weights, 'target_values': target_values, 'delta': delta, 'buys': buys, 'sells': sells, 'gross': gross, 'fraction': traded_fraction, 'unilateral': unilateral_turnover}
    for j, symbol in enumerate(SYMBOLS):
        operation_rows.append({'regla': name, 'ETF': symbol, 'peso_previo_%': 100 * previous_weights[j], 'peso_objetivo_%': 100 * target[j], 'operación_USD': delta[j]})
    turnover_rows.append({'regla': name, 'capital_antes_USD': capital_before, 'compras_USD': buys, 'ventas_USD': sells, 'nocional_total_USD': gross, 'fracción_negociada_%': 100 * traded_fraction, 'rotación_unilateral_%': 100 * unilateral_turnover})

second_risk_exp = tail_metrics(-(second_returns @ second_targets['Mínimo CVaR exponencial']), second_exp)
second_risk_vix = tail_metrics(-(second_returns @ second_targets['Mínimo CVaR VIX']), second_q)
second_controls = {
    'Primera decisión reproducida': first_reproduced,
    'Muestra expansiva admisible y misma para ambos modelos': second_frame.columns == ['session_date', *SYMBOLS] and second_dates[:len(fit_dates)] == fit_dates and len(second_dates) > len(fit_dates) and all(d <= second_fit_end < second_execution for d in second_dates) and second_dates[-1] == second_rebalance['fin_ajuste'],
    'VIX de decisión y alineación x-2': second_rebalance['fecha_vix_decisión'] == second_fit_end and second_decision_vix == vix_by_date[second_fit_end] and all(r['vix_source_date'] == calendar[positions[r['return_date']]-2] for r in second_audit.to_dicts()),
    'Probabilidades normalizadas': all(np.isfinite(p).all() and np.all(p >= 0) and np.isclose(p.sum(), 1, atol=1e-12, rtol=1e-12) for p in (second_exp, second_q)),
    'Entropy pooling factible y KKT': abs(float(second_q @ second_a)) <= 1e-9 and second_q @ second_b <= 1 + 1e-9 and abs(second_gradient[0]) <= 1e-8 and second_theta[1] >= 0 and second_gradient[1] >= -1e-8 and abs(second_theta[1] * second_gradient[1]) <= 1e-8 and np.isfinite(second_kl) and second_kl >= -1e-12,
    'Pesos factibles sin corrección': all(np.isclose(w.sum(), 1, atol=weight_tol, rtol=0) and np.all(w >= -weight_tol) and np.all(w <= cap + weight_tol) for w in second_targets.values()),
    'CVaR concilia con masa exacta y LP': all(np.isclose(x[2], tail, atol=1e-12, rtol=1e-12) and np.isclose(x[0], x[1], atol=1e-12, rtol=1e-12) for x in (second_risk_exp, second_risk_vix)) and abs(second_risk_exp[0] - second_exp_solution.fun) <= CVAR_TOL and abs(second_risk_vix[0] - second_vix_solution.fun) <= CVAR_TOL,
    'Valores previos coinciden con primer periodo': all(np.array_equal(op['before'], first_value_snapshot[name][-1]) and period_dates[-1] == second_execution for name, op in operations.items()),
    'Compras igualan ventas y operaciones suman cero': all(abs(op['buys'] - op['sells']) <= money_tol and abs(op['delta'].sum()) <= money_tol for op in operations.values()),
    'Capital y valores objetivo conservados': all(abs(op['target_values'].sum() - op['capital']) <= money_tol and np.allclose(op['before'] + op['delta'], op['capital'] * second_targets[name], atol=money_tol, rtol=0) for name, op in operations.items()),
    'Nocional y rotaciones concilian': all(np.isclose(op['gross'], op['capital'] * np.abs(second_targets[name] - op['previous_weights']).sum(), atol=money_tol, rtol=0) and np.isclose(op['fraction'], 2 * op['unilateral'], atol=1e-12, rtol=1e-12) for name, op in operations.items()),
    'Primera decisión y periodo preservados': all(np.array_equal(portfolios[name], w) and np.array_equal(accounting[name]['values'], first_value_snapshot[name]) for name, w in first_weight_snapshot.items()) and np.array_equal(p_exp, first_probability_snapshot[0]) and np.array_equal(q_vix, first_probability_snapshot[1]) and eligible_rebalances.equals(first_calendar_snapshot),
}
if not all(second_controls.values()): raise AssertionError([name for name, ok in second_controls.items() if not ok])

print(f'Segunda ejecución: {second_execution}; ajuste: {second_dates[0]} a {second_fit_end}; {second_n} observaciones. VIX {second_fit_end}: {second_decision_vix:.2f}; cortes {second_p25:.2f}/{second_p75:.2f}; grupo {second_group}; entropy pooling {second_case}.')
print(f'KKT: igualdad={second_gradient[0]:.3e}, holgura={second_gradient[1]:.3e}, complementariedad={second_theta[1]*second_gradient[1]:.3e}; KL={second_kl:.6f}.')
with pl.Config(tbl_rows=30, fmt_str_lengths=50):
    display(pl.DataFrame(operation_rows)); display(pl.DataFrame(turnover_rows)); display(pl.DataFrame({'Control': list(second_controls), 'Resultado': ['PASS' if ok else 'FAIL' for ok in second_controls.values()]}))


Segunda ejecución: 2020-02-28; ajuste: 2018-01-04 a 2020-02-27; 540 observaciones. VIX 2020-02-27: 39.16; cortes 12.93/17.91; grupo 3; entropy pooling ACTIVA.
KKT: igualdad=-1.388e-17, holgura=-4.441e-16, complementariedad=-1.991e-16; KL=1.066994.


regla,ETF,peso_previo_%,peso_objetivo_%,operación_USD
str,str,f64,f64,f64
"""Pesos iguales""","""SPY""",12.006974,12.5,4726.374269
"""Pesos iguales""","""VEA""",12.041614,12.5,4394.294873
"""Pesos iguales""","""EEM""",12.546896,12.5,-449.56422
"""Pesos iguales""","""TLT""",13.903292,12.5,-13452.595411
"""Pesos iguales""","""BIL""",13.054681,12.5,-5317.425086
"""Pesos iguales""","""GLD""",12.956282,12.5,-4374.127101
"""Pesos iguales""","""USO""",11.367229,12.5,10859.252042
"""Pesos iguales""","""VNQ""",12.123031,12.5,3613.790634
"""Mínimo CVaR exponencial""","""SPY""",14.291623,15.0,6846.299029


regla,capital_antes_USD,compras_USD,ventas_USD,nocional_total_USD,fracción_negociada_%,rotación_unilateral_%
str,f64,f64,f64,f64,f64,f64
"""Pesos iguales""",958645.234045,23593.711818,23593.711818,47187.423636,4.922303,2.461152
"""Mínimo CVaR exponencial""",966476.233418,55591.668146,55591.668146,111183.336292,11.503991,5.751995
"""Mínimo CVaR VIX""",963052.031754,148980.588807,148980.588807,297961.177614,30.939261,15.46963


Control,Resultado
str,str
"""Primera decisión reproducida""","""PASS"""
"""Muestra expansiva admisible y misma para ambos mod…","""PASS"""
"""VIX de decisión y alineación x-2""","""PASS"""
"""Probabilidades normalizadas""","""PASS"""
"""Entropy pooling factible y KKT""","""PASS"""
"""Pesos factibles sin corrección""","""PASS"""
"""CVaR concilia con masa exacta y LP""","""PASS"""
"""Valores previos coinciden con primer periodo""","""PASS"""
"""Compras igualan ventas y operaciones suman cero""","""PASS"""


La rotación refleja tanto cambios en los objetivos como la deriva de las posiciones. Incluso la regla de pesos iguales requiere operaciones para volver a 12.5% por ETF. Esta comparación supone rebalanceo sin costos y no calcula resultados posteriores.

## 8. ¿Cómo financiamos los costos del rebalanceo?

Ahora aplicamos un costo proporcional por cada dólar comprado o vendido: 0, 5, 10 y 20 puntos básicos por lado. Cada sensibilidad parte de los mismos valores previos del segundo rebalanceo y usa los mismos pesos objetivo; no se encadenan entre sí. Las tasas son supuestos ilustrativos, no cotizaciones de un broker.

El costo sale de la propia cartera. Si $C^-$ es el capital antes de rebalancear, $C^+$ el capital después, $w_j$ el peso objetivo, $V^-_j$ el valor previo y $c$ la tasa por lado, buscamos:

$$C^+ + c\sum_j |C^+ w_j - V^-_j| = C^-.$$

Las operaciones se calculan con el capital posterior a costos, de modo que las ventas financien tanto las compras como el costo. Los pesos objetivo se aplican sobre ese capital posterior; no hay aportaciones ni retiros. No modelamos títulos, órdenes, impuestos o precios de ejecución reales. El caso de cero puntos básicos debe reproducir el rebalanceo bruto anterior.

In [5]:
from scipy.optimize import brentq

# Todas las sensibilidades parten del mismo cierre previo y conservan los objetivos del bloque 7.
cost_levels_bps = [0, 5, 10, 20]
saved_operations = {name: {key: value.copy() if isinstance(value, np.ndarray) else value for key, value in item.items()} for name, item in operations.items()}
saved_targets = {name: weights.copy() for name, weights in second_targets.items()}
saved_first_period = {name: item['values'].copy() for name, item in accounting.items()}
saved_decision_probabilities = (second_exp.copy(), second_q.copy())
saved_decision_calendar = eligible_rebalances.clone()
cost_results = {}
cost_rows = []
for name, target in second_targets.items():
    before_values = operations[name]['before']
    capital_before = operations[name]['capital']
    for bps in cost_levels_bps:
        rate = bps / 10_000.0
        # Resolver el capital neto con las operaciones que él mismo determina.
        def budget_residual(capital_after):
            return capital_after + rate * np.abs(capital_after * target - before_values).sum() - capital_before
        capital_after = capital_before if bps == 0 else brentq(budget_residual, 0.0, capital_before, xtol=1e-8, rtol=1e-14)
        after_values = capital_after * target
        trades = after_values - before_values
        buys = float(np.maximum(trades, 0).sum())
        sells = float(np.maximum(-trades, 0).sum())
        notional = buys + sells
        cost = rate * notional
        cost_results[(name, bps)] = {'capital_before': capital_before, 'capital_after': capital_after, 'after_values': after_values, 'trades': trades, 'buys': buys, 'sells': sells, 'notional': notional, 'cost': cost}
        cost_rows.append({'regla': name, 'costo_por_lado_pb': bps, 'capital_antes_USD': capital_before, 'compras_USD': buys, 'ventas_USD': sells, 'nocional_USD': notional, 'costo_USD': cost, 'capital_posterior_USD': capital_after})

# La tolerancia monetaria es absoluta: USD 1e-6; los pesos se verifican a 1e-9.
cost_money_tol, cost_weight_tol = 1e-6, 1e-9
cost_controls = {
    'Cuatro tasas para cada regla': len(cost_results) == len(second_targets) * len(cost_levels_bps) and all((name, bps) in cost_results for name in second_targets for bps in cost_levels_bps),
    'Capital inicial y valores previos comunes': all(result['capital_before'] == operations[name]['capital'] and np.array_equal(operations[name]['before'], saved_operations[name]['before']) for (name, bps), result in cost_results.items()),
    'Presupuesto autofinanciado': all(abs(result['capital_after'] + result['cost'] - result['capital_before']) <= cost_money_tol and abs(result['sells'] - result['buys'] - result['cost']) <= cost_money_tol for result in cost_results.values()),
    'Costo por compras más ventas': all(abs(result['cost'] - bps / 10_000.0 * (result['buys'] + result['sells'])) <= cost_money_tol for (name, bps), result in cost_results.items()),
    'Valores y pesos objetivo posteriores': all(result['capital_after'] > 0 and np.isfinite(result['after_values']).all() and np.all(result['after_values'] >= 0) and abs(result['after_values'].sum() - result['capital_after']) <= cost_money_tol and np.allclose(result['after_values'] / result['capital_after'], saved_targets[name], atol=cost_weight_tol, rtol=0) for (name, bps), result in cost_results.items()),
    'Sin costos reproduce rebalanceo bruto': all(abs(cost_results[(name, 0)]['capital_after'] - operations[name]['capital']) <= cost_money_tol and np.allclose(cost_results[(name, 0)]['trades'], operations[name]['delta'], atol=cost_money_tol, rtol=0) and cost_results[(name, 0)]['cost'] == 0 for name in second_targets),
    'Costo crece y capital disminuye con la tasa': all(all(cost_results[(name, a)]['cost'] <= cost_results[(name, b)]['cost'] + cost_money_tol and cost_results[(name, a)]['capital_after'] >= cost_results[(name, b)]['capital_after'] - cost_money_tol for a, b in zip(cost_levels_bps, cost_levels_bps[1:])) for name in second_targets),
    'Resultados brutos y entradas preservados': all(np.array_equal(second_targets[name], saved_targets[name]) and np.array_equal(operations[name]['before'], saved_operations[name]['before']) and np.array_equal(operations[name]['delta'], saved_operations[name]['delta']) and np.array_equal(accounting[name]['values'], saved_first_period[name]) for name in second_targets) and np.array_equal(second_exp, saved_decision_probabilities[0]) and np.array_equal(second_q, saved_decision_probabilities[1]) and eligible_rebalances.equals(saved_decision_calendar),
}
if not all(cost_controls.values()): raise AssertionError([name for name, ok in cost_controls.items() if not ok])
print('Segundo rebalanceo:', second_execution, '| tasas en puntos básicos por lado; cada caso parte del mismo capital previo.')
with pl.Config(tbl_rows=20, fmt_str_lengths=55):
    display(pl.DataFrame(cost_rows))
    display(pl.DataFrame({'Control': list(cost_controls), 'Resultado': ['PASS' if ok else 'FAIL' for ok in cost_controls.values()]}))


Segundo rebalanceo: 2020-02-28 | tasas en puntos básicos por lado; cada caso parte del mismo capital previo.


regla,costo_por_lado_pb,capital_antes_USD,compras_USD,ventas_USD,nocional_USD,costo_USD,capital_posterior_USD
str,i64,f64,f64,f64,f64,f64,f64
"""Pesos iguales""",0,958645.234045,23593.711818,23593.711818,47187.423636,0.0,958645.234045
"""Pesos iguales""",5,958645.234045,23581.914962,23605.508674,47187.423636,23.593712,958621.640333
"""Pesos iguales""",10,958645.234045,23570.118106,23617.30553,47187.423636,47.187424,958598.046621
"""Pesos iguales""",20,958645.234045,23546.524394,23640.899242,47187.423636,94.374847,958550.859198
"""Mínimo CVaR exponencial""",0,966476.233418,55591.668146,55591.668146,111183.336292,0.0,966476.233418
"""Mínimo CVaR exponencial""",5,966476.233418,55562.073401,55617.663269,111179.73667,55.589868,966420.64355
"""Mínimo CVaR exponencial""",10,966476.233418,55532.480572,55643.656709,111176.137282,111.176137,966365.057281
"""Mínimo CVaR exponencial""",20,966476.233418,55473.300662,55695.638541,111168.939203,222.337878,966253.89554
"""Mínimo CVaR VIX""",0,963052.031754,148980.588807,148980.588807,297961.177614,0.0,963052.031754


Control,Resultado
str,str
"""Cuatro tasas para cada regla""","""PASS"""
"""Capital inicial y valores previos comunes""","""PASS"""
"""Presupuesto autofinanciado""","""PASS"""
"""Costo por compras más ventas""","""PASS"""
"""Valores y pesos objetivo posteriores""","""PASS"""
"""Sin costos reproduce rebalanceo bruto""","""PASS"""
"""Costo crece y capital disminuye con la tasa""","""PASS"""
"""Resultados brutos y entradas preservados""","""PASS"""


Los costos reducen el capital disponible para alcanzar los pesos objetivo. Al autofinanciar el rebalanceo, las ventas superan a las compras por el costo pagado. Esta sensibilidad describe una sola fecha y no evalúa todavía rendimientos posteriores ni costos reales de ejecución.

## 9. ¿Cómo se comportan las tres reglas durante todo el recorrido?

Reconstruimos cada decisión con la historia admisible de esa fecha y después seguimos las posiciones hasta el corte. Las cuatro tasas parten por separado de USD 1,000,000 e incluyen el costo de entrada desde efectivo; no se liquida la cartera al final. Los retornos total-return suponen reinversión según la convención del proveedor.

El CAGR usa días naturales transcurridos desde la primera ejecución hasta el corte, con año de 365.25 días. La volatilidad usa desviación poblacional de retornos diarios netos multiplicada por la raíz de 252; el costo de entrada se incorpora al primer retorno diario. El drawdown incluye el capital previo a ese costo. Los costos acumulados son pagos explícitos y no la diferencia entre capital bruto y neto, que también incorpora su efecto posterior.

Es una evaluación retrospectiva: el universo fue seleccionado después de observar la historia, la muestra ya se exploró, los datos pueden revisarse y la ejecución al cierre es idealizada.

In [6]:
from time import perf_counter

# La función de decisión recibe sólo datos de ajuste y el estado admisible de ese origen.
def build_decision(history_returns, history_vix, history_positions, current_vix, half_life, tail_probability, max_weight):
    n, assets = history_returns.shape
    if n < M or assets != len(SYMBOLS) or len(history_vix) != n or len(history_positions) != n or not np.isfinite(history_returns).all() or not np.isfinite(history_vix).all() or not np.isfinite(current_vix):
        raise ValueError('Insumos de decisión incompletos o inválidos.')
    ages = history_positions[-1] - history_positions
    prior = np.exp(-np.log(2.0) * ages / half_life); prior /= prior.sum()
    ordered = np.sort(history_vix)
    lower, upper = [float(ordered[int(np.ceil(n * alpha)) - 1]) for alpha in (0.25, 0.75)]
    if not lower < upper: raise ValueError('Cortes VIX no distintos.')
    groups = np.where(history_vix <= lower, 1, np.where(history_vix <= upper, 2, 3))
    state = 1 if current_vix <= lower else (2 if current_vix <= upper else 3)
    selected = history_vix[groups == state]
    if len(selected) == 0: raise ValueError('Grupo VIX vacío.')
    mean = float(selected.mean()); variance = float(np.mean((selected-mean)**2))
    if variance <= 0: raise ValueError('Varianza VIX crisp no positiva.')
    x = (history_vix-mean) / np.sqrt(variance); x2 = x*x; log_prior = np.log(prior)
    def dual(theta):
        log_raw = log_prior-theta[0]*x-theta[1]*x2; log_q = log_raw-logsumexp(log_raw); q = np.exp(log_q)
        return float(logsumexp(log_raw)+theta[1]), np.array([-q@x, 1-q@x2]), q, log_q
    def hessian(theta):
        q = dual(theta)[2]; features = np.column_stack((x,x2)); centered = features-q@features
        return (centered*q[:,None]).T@centered
    theta, state_case = solve_entropy_dual(dual, hessian)
    _, grad, adjusted, log_adjusted = dual(theta)
    kl_value = float(adjusted @ (log_adjusted-log_prior))
    if not (np.all(prior > 0) and np.isfinite(prior).all() and np.isfinite(adjusted).all() and np.all(adjusted >= 0) and np.isclose(prior.sum(), 1, atol=1e-12, rtol=1e-12) and np.isclose(adjusted.sum(), 1, atol=1e-12, rtol=1e-12) and abs(float(adjusted@x)) <= 1e-9 and adjusted@x2 <= 1+1e-9 and abs(grad[0]) <= 1e-8 and theta[1] >= 0 and grad[1] >= -1e-8 and abs(theta[1]*grad[1]) <= 1e-8 and np.isfinite(kl_value) and kl_value >= -1e-12):
        raise AssertionError('Probabilidades o KKT de entropy pooling no válidos.')
    matrix = hstack([csr_matrix(-history_returns), csr_matrix(-np.ones((n,1))), -eye(n, format='csr')], format='csr')
    investment = csr_matrix(np.r_[np.ones(assets), np.zeros(1+n)].reshape(1,-1))
    variable_bounds = [(0.0,max_weight)]*assets + [(None,None)] + [(0.0,None)]*n
    exp_lp = solve_cvar(prior, matrix, investment, variable_bounds)
    vix_lp = solve_cvar(adjusted, matrix, investment, variable_bounds)
    weights = {'Pesos iguales': np.full(assets, 1/assets), 'Mínimo CVaR exponencial': exp_lp.x[:assets].copy(), 'Mínimo CVaR VIX': vix_lp.x[:assets].copy()}
    for name, w in weights.items():
        if not (np.isfinite(w).all() and abs(w.sum()-1) <= 1e-9 and np.all(w >= -1e-9) and np.all(w <= max_weight+1e-9)):
            raise AssertionError(f'Pesos no factibles: {name}.')
    for result, probabilities, name in ((exp_lp, prior, 'Mínimo CVaR exponencial'), (vix_lp, adjusted, 'Mínimo CVaR VIX')):
        losses = -(history_returns @ weights[name]); cvar, independent, mass = tail_metrics(losses, probabilities)
        if not (abs(cvar-result.fun) <= 1e-8 and abs(cvar-independent) <= 1e-12 and abs(mass-tail_probability) <= 1e-12 and np.all(result.x[assets+1:] >= losses-result.x[assets]-1e-9)):
            raise AssertionError(f'Conciliación CVaR fallida: {name}.')
    diag = {'observaciones': n, 'p25': lower, 'p75': upper, 'vix_decisión': current_vix, 'grupo': state, 'n_crisp': len(selected), 'media_objetivo': mean, 'varianza_límite': variance, 'caso': state_case, 'eta': float(theta[1]), 'residuo_media': float(grad[0]), 'holgura': float(grad[1]), 'complementariedad': float(theta[1]*grad[1]), 'kl': kl_value, 'neff_exp': float(1/(prior@prior)), 'neff_vix': float(1/(adjusted@adjusted)), 'underflow': int(np.count_nonzero(adjusted == 0)), 'cvar_exp': float(exp_lp.fun), 'cvar_vix': float(vix_lp.fun)}
    return weights, diag

started = perf_counter()
prior_examples = {name: w.copy() for name, w in portfolios.items()}
second_examples = {name: w.copy() for name, w in second_targets.items()}
prior_accounting = {name: item['values'].copy() for name, item in accounting.items()}
prior_costs = {(name,bps): item['capital_after'] for (name,bps), item in cost_results.items()}
decision_dates = eligible_rebalances['fecha_ejecución'].to_list()
decisions = {}; decision_diagnostics = []; decision_failures = []
for row in eligible_rebalances.to_dicts():
    decision_date = row['fecha_ejecución']; end = row['fin_información']
    try:
        available = audit.filter((pl.col('alignment_status') == 'ALIGNED') & (pl.col('return_date') <= end))
        dates = available['return_date'].to_list()
        rows = returns.filter(pl.col('session_date').is_in(dates))
        if len(dates) != row['observaciones'] or rows['session_date'].to_list() != dates or dates[-1] > end or end >= decision_date or row['fecha_vix_decisión'] != end or row['vix_decisión'] != vix_by_date[end]:
            raise AssertionError('Fechas o estado admisible incongruentes.')
        history_positions = np.array([positions[d] for d in dates])
        w, diagnostics = build_decision(rows.select(SYMBOLS).to_numpy(), available['vix_close'].to_numpy(), history_positions, row['vix_decisión'], TAU, tail, cap)
        decisions[decision_date] = w
        decision_diagnostics.append({'fecha': decision_date, 'fin_ajuste': end, 'inicio_ajuste': dates[0], **diagnostics})
    except Exception as exc:
        decision_failures.append({'fecha': decision_date, 'causa': f'{type(exc).__name__}: {exc}'})
        raise RuntimeError(f'Decisión fallida {decision_date}: {type(exc).__name__}: {exc}') from exc

for name in portfolios:
    if not np.allclose(decisions[decision_dates[0]][name], prior_examples[name], atol=1e-9, rtol=0) or not np.allclose(decisions[decision_dates[1]][name], second_examples[name], atol=1e-9, rtol=0):
        raise AssertionError(f'Las dos decisiones previas no se reproducen: {name}.')
decision_table = pl.DataFrame([{'fecha': day, 'regla': name, 'ETF': symbol, 'peso_objetivo': float(decisions[day][name][j])} for day in decision_dates for name in portfolios for j,symbol in enumerate(SYMBOLS)])
diagnostic_table = pl.DataFrame(decision_diagnostics)

# Cada sensibilidad tiene sus propios valores y costos, pero recibe los mismos pesos objetivo por regla.
evaluation_dates = calendar[positions[decision_dates[0]]+1:]
evaluation_frame = returns.filter((pl.col('session_date') > decision_dates[0]) & (pl.col('session_date') <= cut))
if evaluation_frame['session_date'].to_list() != evaluation_dates: raise AssertionError('Sesiones de evaluación incompletas.')
evaluation_returns = evaluation_frame.select(SYMBOLS).to_numpy()
runs = {}; daily_rows = []; execution_rows = []; position_rows = []; factor_errors = []
for name in portfolios:
    for bps in cost_levels_bps:
        rate = bps / 10000.0; start_capital = 1_000_000.0
        invested = start_capital / (1 + rate); entry_cost = start_capital - invested
        values = invested * decisions[decision_dates[0]][name]
        cash = [start_capital]; daily_net_returns = []; total_cost = entry_cost; total_notional = invested
        entries = [{'fecha': decision_dates[0], 'tipo': 'entrada', 'antes': start_capital, 'después': invested, 'compras': invested, 'ventas': 0.0, 'nocional': invested, 'costo': entry_cost}]
        for j, symbol in enumerate(SYMBOLS):
            position_rows.append({'fecha': decision_dates[0], 'regla': name, 'pb': bps, 'ETF': symbol, 'peso_antes': 0.0, 'peso_después': float(decisions[decision_dates[0]][name][j]), 'valor_antes': 0.0, 'valor_después': float(values[j])})
        last_post = invested; period_factor = 1.0; factor_capital = invested
        for day, asset_returns in zip(evaluation_dates, evaluation_returns):
            previous_total = float(values.sum()); previous_weights = values / previous_total
            one_day_factor = float(1.0 + previous_weights @ asset_returns)
            values = values * (1.0 + asset_returns); before = float(values.sum()); period_factor *= one_day_factor
            if day in decisions:
                target = decisions[day][name]; before_values = values.copy(); pre_weights = before_values / before
                def residual(after): return after + rate * np.abs(after*target-before_values).sum() - before
                after = before if bps == 0 else brentq(residual, 0.0, before, xtol=1e-8, rtol=1e-14)
                values = after * target; delta = values-before_values
                buys = float(np.maximum(delta,0).sum()); sells = float(np.maximum(-delta,0).sum()); notional = buys+sells; cost = rate*notional
                if abs(after+cost-before) > 1e-6 or abs(sells-buys-cost) > 1e-6 or abs(values.sum()-after) > 1e-6:
                    raise AssertionError(f'Contabilidad fallida: {day}, {name}, {bps} pb.')
                factor_capital = factor_capital*period_factor-cost; factor_errors.append(abs(factor_capital-after)); period_factor = 1.0; last_post = after
                total_cost += cost; total_notional += notional
                entries.append({'fecha': day, 'tipo': 'rebalanceo', 'antes': before, 'después': after, 'compras': buys, 'ventas': sells, 'nocional': notional, 'costo': cost})
                for j, symbol in enumerate(SYMBOLS):
                    position_rows.append({'fecha': day, 'regla': name, 'pb': bps, 'ETF': symbol, 'peso_antes': float(pre_weights[j]), 'peso_después': float(target[j]), 'valor_antes': float(before_values[j]), 'valor_después': float(values[j])})
            current = float(values.sum())
            if not (np.isfinite(values).all() and np.all(values >= -1e-8) and current > 0 and abs((values/current).sum()-1) <= 1e-12):
                raise AssertionError(f'Posiciones inválidas: {day}, {name}, {bps} pb.')
            daily_net_returns.append(current / cash[-1] - 1.0); cash.append(current)
            daily_rows.append({'fecha': day, 'regla': name, 'pb': bps, 'capital': current, 'retorno_diario': daily_net_returns[-1]})
        factor_capital *= period_factor; factor_errors.append(abs(factor_capital-cash[-1]))
        capital_path = np.array(cash); daily_net_returns = np.array(daily_net_returns)
        # Incluimos el punto previo y el posterior al costo de entrada en la misma fecha.
        drawdown_capital = np.r_[capital_path[0], invested, capital_path[1:]]
        drawdowns = drawdown_capital/np.maximum.accumulate(drawdown_capital)-1.0
        runs[(name,bps)] = {'capital': capital_path, 'capital_after_entry': invested, 'returns': daily_net_returns, 'drawdown': drawdowns, 'final_values': values.copy(), 'cost': total_cost, 'notional': total_notional, 'executions': entries}
        execution_rows.extend({'regla': name, 'pb': bps, **entry} for entry in entries)

daily_table = pl.DataFrame(daily_rows); execution_table = pl.DataFrame(execution_rows); positions_table = pl.DataFrame(position_rows)
elapsed_seconds = perf_counter()-started
years = (cut-decision_dates[0]).days / 365.25
main_rows = []; sensitivity_rows = []
for name in portfolios:
    item = runs[(name,5)]; final = item['capital'][-1]
    main_rows.append({'regla': name, 'capital_final_USD': final, 'retorno_neto_%': 100*(final/1_000_000-1), 'CAGR_%': 100*((final/1_000_000)**(1/years)-1), 'vol_diaria_anualizada_%': 100*np.std(item['returns'], ddof=0)*np.sqrt(252), 'drawdown_máximo_%': 100*item['drawdown'].min(), 'costos_pagados_USD': item['cost'], 'nocional_total_USD': item['notional']})
    for bps in cost_levels_bps:
        scenario = runs[(name,bps)]
        sensitivity_rows.append({'regla': name, 'pb_por_lado': bps, 'capital_final_USD': scenario['capital'][-1], 'drawdown_máximo_%': 100*scenario['drawdown'].min()})

# Comprobaciones de reproducción y conciliación; el ejemplo local de costos usa sus propios valores previos originales.
example_cost_ok = True
for name in portfolios:
    for bps in cost_levels_bps:
        example = cost_results[(name,bps)]; before = operations[name]['before']; target = second_targets[name]; rate = bps/10000.0
        f = lambda after: after + rate*np.abs(after*target-before).sum() - before.sum()
        after = float(before.sum()) if bps == 0 else brentq(f, 0.0, float(before.sum()), xtol=1e-8, rtol=1e-14)
        example_cost_ok &= abs(after-example['capital_after']) <= 1e-6
backtest_controls = {
    '81 decisiones con fechas exactas': len(decisions) == 81 and list(decisions) == decision_dates and len(decision_diagnostics) == 81 and not decision_failures,
    'Mismos días, sin omisiones ni duplicados': len(evaluation_dates) == len(set(evaluation_dates)) and evaluation_dates == sorted(evaluation_dates) and all(daily_table.filter((pl.col('regla') == name) & (pl.col('pb') == bps))['fecha'].to_list() == evaluation_dates for name in portfolios for bps in cost_levels_bps),
    'Dos primeras decisiones reproducidas': all(np.allclose(decisions[decision_dates[0]][name], prior_examples[name], atol=1e-9, rtol=0) and np.allclose(decisions[decision_dates[1]][name], second_examples[name], atol=1e-9, rtol=0) for name in portfolios),
    'Primer periodo bruto reproducido': all(np.allclose(runs[(name,0)]['capital'][:len(period_dates)+1], accounting[name]['capital'], atol=1e-6, rtol=1e-12) for name in portfolios),
    'Ejemplo local de costos reproducido': example_cost_ok,
    'Ajustes y objetivos factibles en cada fecha': len(decision_diagnostics) == 81 and all(abs(d['residuo_media']) <= 1e-8 and d['holgura'] >= -1e-8 and abs(d['complementariedad']) <= 1e-8 for d in decision_diagnostics) and all(abs(w.sum()-1) <= 1e-9 and np.all(w >= -1e-9) and np.all(w <= cap+1e-9) for per_date in decisions.values() for w in per_date.values()),
    '81 ejecuciones por regla y tasa': all([e['fecha'] for e in runs[(name,bps)]['executions']] == decision_dates for name in portfolios for bps in cost_levels_bps),
    'Capital y posiciones válidos en todos los recorridos': all(np.isfinite(r['capital']).all() and np.all(r['capital'] > 0) and np.all(r['final_values'] >= -1e-8) for r in runs.values()) and positions_table.filter((pl.col('peso_antes') < -1e-9) | (pl.col('peso_después') < -1e-9)).height == 0,
    'Costos entrada y cierre sin liquidación': all(abs(runs[(name,bps)]['executions'][0]['después'] - 1_000_000/(1+bps/10000)) <= 1e-6 and abs(runs[(name,bps)]['final_values'].sum()-runs[(name,bps)]['capital'][-1]) <= 1e-6 and runs[(name,bps)]['executions'][-1]['fecha'] == decision_dates[-1] < cut for name in portfolios for bps in cost_levels_bps),
    'Conciliación por factores y costos': max(factor_errors) <= 1e-5,
    'Cero pb sin costos y composición bruta': all(runs[(name,0)]['cost'] == 0 and np.allclose(runs[(name,0)]['capital'][1:], 1_000_000*np.cumprod(1+runs[(name,0)]['returns']), atol=1e-5, rtol=1e-12) for name in portfolios),
    'Ejemplos anteriores preservados': all(np.array_equal(portfolios[name],prior_examples[name]) and np.array_equal(second_targets[name],second_examples[name]) and np.array_equal(accounting[name]['values'],prior_accounting[name]) for name in portfolios) and all(cost_results[key]['capital_after'] == value for key,value in prior_costs.items()),
}
if not all(backtest_controls.values()): raise AssertionError([name for name, ok in backtest_controls.items() if not ok])

print(f'Decisiones válidas: {len(decisions)}; fallidas: {len(decision_failures)}; ejecución: {elapsed_seconds:.2f} s. Primera ejecución {decision_dates[0]}; última {decision_dates[-1]}; corte parcial {cut}; sesiones evaluadas: {len(evaluation_dates)}.')
print(f'CAGR: {years:.6f} años de 365.25 días; error máximo de conciliación por factores: USD {max(factor_errors):.3e}.')
with pl.Config(tbl_rows=20, fmt_str_lengths=55, float_precision=4):
    display(pl.DataFrame(main_rows)); display(pl.DataFrame(sensitivity_rows)); display(pl.DataFrame({'Control': list(backtest_controls), 'Resultado': ['PASS' if ok else 'FAIL' for ok in backtest_controls.values()]}))

colors = {'Pesos iguales': '#777777', 'Mínimo CVaR exponencial': '#B55A3C', 'Mínimo CVaR VIX': '#397B72'}
for title, field, label in [('Capital neto a 5 pb por lado', 'capital', 'Capital (USD)'), ('Drawdown neto a 5 pb por lado', 'drawdown', 'Drawdown (%)')]:
    chart = go.Figure()
    for name in portfolios:
        values = runs[(name,5)][field]
        values = values*100 if field == 'drawdown' else np.r_[values[0], runs[(name,5)]['capital_after_entry'], values[1:]]
        chart_dates = [decision_dates[0], decision_dates[0], *evaluation_dates]
        chart.add_scatter(x=chart_dates, y=values, mode='lines', name=name, line=dict(color=colors[name]), hovertemplate='%{x|%Y-%m-%d}<br>%{y:,.2f}<extra>%{fullData.name}</extra>')
    chart.add_vrect(x0=str(decision_dates[-1]), x1=str(cut), fillcolor='#D9CDBA', opacity=0.22, line_width=0, layer='below')
    chart.update_layout(title=title+' — periodo final parcial sombreado', xaxis_title='Fecha', yaxis_title=label, paper_bgcolor='#FFF8E7', plot_bgcolor='#FFF8E7', font_color='#2D2A26', legend=dict(orientation='h', y=1.02, x=0, yanchor='bottom'), margin=dict(l=80,r=35,t=110,b=60))
    chart.update_xaxes(gridcolor='#E7DDCC'); chart.update_yaxes(gridcolor='#E7DDCC'); chart.show()


Decisiones válidas: 81; fallidas: 0; ejecución: 8.38 s. Primera ejecución 2020-01-31; última 2026-09-30; corte parcial 2026-10-09; sesiones evaluadas: 1681.
CAGR: 6.688569 años de 365.25 días; error máximo de conciliación por factores: USD 1.024e-08.


regla,capital_final_USD,retorno_neto_%,CAGR_%,vol_diaria_anualizada_%,drawdown_máximo_%,costos_pagados_USD,nocional_total_USD
str,f64,f64,f64,f64,f64,f64,f64
"""Pesos iguales""",1814299.4813,81.4299,9.3149,12.4785,-25.2310,2159.7530,4319506.0134
"""Mínimo CVaR exponencial""",1694134.5968,69.4135,8.2006,11.3637,-22.3488,2184.0669,4368133.7336
"""Mínimo CVaR VIX""",1707196.0840,70.7196,8.3249,11.3284,-22.2404,5553.4347,11106869.4224


regla,pb_por_lado,capital_final_USD,drawdown_máximo_%
str,i64,f64,f64
"""Pesos iguales""",0,1817681.3631,-25.2292
"""Pesos iguales""",5,1814299.4813,-25.2310
"""Pesos iguales""",10,1810924.4723,-25.2329
"""Pesos iguales""",20,1804194.9945,-25.2365
"""Mínimo CVaR exponencial""",0,1697325.9116,-22.3351
"""Mínimo CVaR exponencial""",5,1694134.5968,-22.3488
"""Mínimo CVaR exponencial""",10,1690949.7826,-22.3626
"""Mínimo CVaR exponencial""",20,1684599.5969,-22.3900
"""Mínimo CVaR VIX""",0,1715046.0468,-22.2050


Control,Resultado
str,str
"""81 decisiones con fechas exactas""","""PASS"""
"""Mismos días, sin omisiones ni duplicados""","""PASS"""
"""Dos primeras decisiones reproducidas""","""PASS"""
"""Primer periodo bruto reproducido""","""PASS"""
"""Ejemplo local de costos reproducido""","""PASS"""
"""Ajustes y objetivos factibles en cada fecha""","""PASS"""
"""81 ejecuciones por regla y tasa""","""PASS"""
"""Capital y posiciones válidos en todos los recorridos""","""PASS"""
"""Costos entrada y cierre sin liquidación""","""PASS"""


A 5 pb por lado, pesos iguales terminó con un retorno neto de 81.43% y un drawdown máximo de −25.23%. Las reglas de mínimo CVaR exponencial y condicionado por VIX terminaron con retornos de 69.41% y 70.72%, y drawdowns de −22.35% y −22.24%, respectivamente. En esta muestra, las dos últimas combinaron menor crecimiento con menor drawdown.

El condicionado pagó aproximadamente USD 5,553 de costos explícitos, frente a USD 2,184 para el exponencial y USD 2,160 para pesos iguales. Esto refleja una mayor actividad negociada bajo los supuestos del ejercicio. Las diferencias son retrospectivas y no demuestran superioridad estadística. El universo fue seleccionado retrospectivamente, la muestra ya se exploró, los datos pueden revisarse y la ejecución al cierre es idealizada.